In [0]:
import pandas as pd
import os

# 1. RUTA: carpeta 'Datos' del repositorio clonado en tu Workspace
folder_path = "/Workspace/Users/qbrandon2003@gmail.com/Proyecto_SQL_Caso_Cancelaciones_Hotel_BQH/Datos"

# 2. ESQUEMA: destino en hotel_db
catalog_schema = "cursodataacademy.hotel_db"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_schema}")

csv_files = [f for f in os.listdir(folder_path) if f.lower().endswith(".csv")]
print(f"Se encontraron {len(csv_files)} archivos CSV:\n")

for csv_file in sorted(csv_files):
    file_path = os.path.join(folder_path, csv_file)
    table_name = csv_file.replace(".csv", "").lower()   
    full_table_name = f"{catalog_schema}.{table_name}"

    # keep_default_na=False conserva 'NA' y 'NULL' como texto (children, agent, company)
    df_pandas = pd.read_csv(file_path, encoding="utf-8-sig", sep=",",
                        skipinitialspace=True, low_memory=False)

    # Reemplaza espacios y caracteres problemáticos por guiones bajos
    df_pandas.columns = df_pandas.columns.str.replace(r'[ ,;{}()\n\t=]', '_', regex=True)

    # Crear DataFrame de Spark y guardar como tabla Delta
    df_spark = spark.createDataFrame(df_pandas)
    df_spark.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_table_name)

    row_count = df_spark.count()
    col_count = len(df_spark.columns)
    print(f"✅ {csv_file} → {full_table_name} ({row_count} filas, {col_count} columnas)")

print(f"\n✨ Carga completa: {len(csv_files)} tablas creadas en {catalog_schema}")



In [0]:
%sql
ALTER TABLE cursodataacademy.hotel_db.tb_reservas ADD COLUMNS (arrival_date DATE);

In [0]:
%sql
UPDATE cursodataacademy.hotel_db.tb_reservas
SET arrival_date = TRY_TO_DATE(CONCAT(arrival_date_year,'-',arrival_date_month,'-',arrival_date_day_of_month),'yyyy-MMMM-d');

In [0]:
%sql
SELECT COUNT(*) AS total, COUNT(arrival_date) AS con_fecha
FROM cursodataacademy.hotel_db.tb_reservas;